In [ ]:
import requests
from bs4 import BeautifulSoup
import getpass
import os
from langchain_openrouter import ChatOpenRouter

from langchain_core.tools import tool

from langchain.agents import create_agent
import pprint

In [ ]:
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

In [ ]:
model = ChatOpenRouter(
    model="openrouter/hunter-alpha",
    temperature=0.01,
    max_tokens=1024,
    max_retries=2,
)

In [ ]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to Malayalam. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = model.invoke(messages)
ai_msg.content

In [ ]:
@tool
def load_webpage(url: str) -> str:
    """
    Load webpage content from a URL and return cleaned text.
    
    Args:
        url (str): The URL of the webpage to load.
        
    Returns:
        str: The cleaned text content of the webpage.
    
    """
    # Use Mozilla Firefox user agent
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    # Fetch the webpage
    response = requests.get(url, headers=headers, timeout=10)

    if response.status_code != 200:
        return f"Error: Unable to fetch page (status {response.status_code})"
    # Parse the HTML content
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract text from paragraphs
    paragraphs = soup.find_all("p")
    text = " ".join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])

    print("Extracted length:", len(text))

    if not text:
        return "Error: No content extracted from webpage"

    return text[:5000]

In [ ]:
@tool
def summarize_text(text: str) -> str:
    """
    Summarize the given text into a concise summary.
    
    Args: 
    text (str): The text to summarize.
    
    Returns: 
    str: The summarized text.   
    """
    return model.invoke(f"Summarize this:\n\n{text}").content


In [ ]:
system_prompt = """
You are an AI agent.

Steps:
1. Extract URL from user input
2. Use load_webpage tool to fetch content
3. If content is empty or error, retry once
4. Then summarize using summarize_text
5. Return clean final answer
"""

In [ ]:
agent = create_agent(
    model=model,
    tools=[load_webpage, summarize_text],
    system_prompt=system_prompt
)

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Find a summary of https://en.wikipedia.org/wiki/Kerala"
        }
    ]
})

print(response)

In [ ]:
type(response)

### Agent Thinking → AIMessage (reasoning)
### Tool Execution → ToolMessage
### Final Answer → AIMessage (last message)

In [ ]:
pprint.pprint(response)

In [ ]:
real_summary = response["messages"][-1].content
real_summary

## References

1. LangChain (Core)
    * LangChain Agents Docs: https://docs.langchain.com/oss/python/langchain/agents

    * LangChain Tools: https://docs.langchain.com/oss/python/langchain/tools

    * LangChain Overview: https://docs.langchain.com 

2. Agentic AI Concepts
    * Single vs Multi-Agent Systems: https://medium.com/data-science-collective/agentic-ai-single-vs-multi-agent-systems-e5c8b0e3cb28

    * Introduction to Agentic AI and Its Design Patterns: https://lekha-bhan88.medium.com/introduction-to-agentic-ai-and-its-design-patterns-af8b7b3ef738

3. Web Scrapping
    * BeautifulSoup Docs: https://www.crummy.com/software/BeautifulSoup/bs4/doc/

4. OpenRouter
    * OpenRouter Docs: https://openrouter.ai/docs

    * Model List: https://openrouter.ai/models
